In [ ]:

import re
import html
from urllib.parse import urlparse

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC


# --------------------------------------------------
# 1. Load the cleaned Phase 2 dataset
# --------------------------------------------------

DATA_FILE = "fakenewsnet_cleaned_final.csv"

SEEDS = list(range(10))
TEST_SIZE = 0.2

df = pd.read_csv(DATA_FILE)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())


# --------------------------------------------------
# 2. Clean and prepare the data
# --------------------------------------------------

df = df.copy()

df["title"] = df["title"].fillna("").astype(str)
df["title"] = df["title"].apply(html.unescape).str.strip()
df["title"] = df["title"].str.replace(r"\s+", " ", regex=True)

# representation: 1 = fake, 0 = real
df["y"] = pd.to_numeric(df["representation"], errors="coerce")

df = df[df["y"].isin([0, 1])]
df["y"] = df["y"].astype(int)

# Remove rows without titles
df = df[df["title"] != ""].copy()

# Remove titles that have conflicting labels
title_labels = df.groupby(
    df["title"].str.lower()
)["y"].transform("nunique")

df = df[title_labels == 1].copy()

# Remove duplicate titles
df["title_key"] = df["title"].str.lower()
df = df.drop_duplicates("title_key")

df = df.drop(columns=["title_key"]).reset_index(drop=True)


# --------------------------------------------------
# 3. Extract publisher domains
# --------------------------------------------------

def get_publisher(url):
    if not isinstance(url, str) or not url.strip():
        return None

    url = url.strip()

    if not url.startswith(("http://", "https://")):
        url = "https://" + url

    host = urlparse(url).netloc.lower().split(":")[0]

    if host.startswith("www."):
        host = host[4:]

    if not host:
        return None

    parts = host.split(".")

    # Handle common domains such as bbc.co.uk
    if (
        len(parts) >= 3
        and len(parts[-1]) == 2
        and parts[-2] in ("co", "com", "org", "net", "gov", "ac")
    ):
        return ".".join(parts[-3:])

    if len(parts) >= 2:
        return ".".join(parts[-2:])

    return host


# Always use the main domain from the URL, so that e.g. en.wikipedia.org
# and wikipedia.org count as the same publisher
df["publisher"] = df["news_url"].apply(get_publisher)

df = df.dropna(subset=["publisher"]).copy()
df = df[
    df["publisher"].astype(str).str.strip() != ""
].reset_index(drop=True)

# Remove junk: very short articles and "publishers" that are not news sites
NON_NEWS = {
    "wikipedia.org", "celebritynetworth.com", "instagram.com", "twitter.com",
    "facebook.com", "youtube.com", "amazon.com", "politifact.com"
}
before = len(df)
df = df[~df["publisher"].isin(NON_NEWS)]

if "article_text" in df.columns:
    n_words = df["article_text"].fillna("").astype(str).str.split().str.len()
    df = df[n_words >= 50]

df = df.reset_index(drop=True)
print("Removed", before - len(df), "junk rows (short text / non-news sites)")

print("Rows after cleaning:", len(df))
print("Unique publishers:", df["publisher"].nunique())
print("Fake news:", (df["y"] == 1).sum())
print("Real news:", (df["y"] == 0).sum())


# --------------------------------------------------
# 4. Mask publisher-related clues in the text
# --------------------------------------------------

def mask_publisher(text):
    text = str(text)

    # Remove website-domain names
    text = re.sub(
        r"\b[\w-]+\.(com|net|org|co\.uk|info|tv)\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # Remove short publisher tags at the end of a title
    text = re.sub(r"\s[\|–—-]\s*\S+(\s\S+)?$", " ", text)

    return re.sub(r"\s+", " ", text).strip()


df["title_masked"] = df["title"].apply(mask_publisher)

# Include article bodies when available
TEXT_VERSIONS = [
    ("title", "title_plain"),
    ("title_masked", "title_masked")
]

if "article_text" in df.columns:
    df["article_text"] = (
        df["article_text"].fillna("").astype(str).str.strip()
    )

    # Keep rows with usable body text for body experiments
    df["article_text_masked"] = df["article_text"].apply(mask_publisher)

    TEXT_VERSIONS.append(("article_text", "body_plain"))
    TEXT_VERSIONS.append(("article_text_masked", "body_masked"))

print("Text experiments:", [name for col, name in TEXT_VERSIONS])


# --------------------------------------------------
# 5. Create random and publisher-held-out splits
# --------------------------------------------------

def random_split(data, seed):
    return train_test_split(
        data,
        test_size=TEST_SIZE,
        random_state=seed,
        stratify=data["y"]
    )


def publisher_split(data, seed):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=seed
    )

    train_idx, test_idx = next(
        splitter.split(data, groups=data["publisher"])
    )

    train = data.iloc[train_idx]
    test = data.iloc[test_idx]

    # Make sure the same publisher is not in both sets
    assert set(train["publisher"]).isdisjoint(
        set(test["publisher"])
    )

    return train, test


# --------------------------------------------------
# 6. Select a model
# --------------------------------------------------

def get_model(name):
    if name == "Logistic Regression":
        return LogisticRegression(
            C=1.0,
            max_iter=1000,
            class_weight="balanced"
        )

    if name == "Linear SVM":
        return LinearSVC(
            C=0.5,
            class_weight="balanced"
        )

    if name == "Naive Bayes":
        return ComplementNB(alpha=0.5)

    if name == "Majority baseline":
        # always predicts the most common class (real news)
        return DummyClassifier(strategy="most_frequent")


# --------------------------------------------------
# 7. Train and evaluate a model
# --------------------------------------------------

def train_and_test(model_name, train, test, text_col):
    # Fit the vectorizer only on training data
    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=50000
    )

    X_train = vectorizer.fit_transform(train[text_col])
    X_test = vectorizer.transform(test[text_col])

    model = get_model(model_name)
    model.fit(X_train, train["y"])

    predictions = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(
        test["y"], predictions, labels=[0, 1]
    ).ravel()

    return {
        "n_test": len(test),
        "test_fake_rate": test["y"].mean(),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "acc": accuracy_score(test["y"], predictions),
        "prec": precision_score(
            test["y"], predictions, zero_division=0
        ),
        "rec": recall_score(
            test["y"], predictions, zero_division=0
        ),
        "f1": f1_score(
            test["y"], predictions, zero_division=0
        )
    }


# --------------------------------------------------
# 8. Run the experiments
# --------------------------------------------------

MODELS = [
    "Logistic Regression",
    "Linear SVM",
    "Naive Bayes",
    "Majority baseline"
]

results = []

for seed in SEEDS:
    for split_name, split_function in [
        ("random", random_split),
        ("held-out", publisher_split)
    ]:

        train, test = split_function(df, seed)

        # Skip a split if training does not contain both classes
        if train["y"].nunique() < 2:
            print("Skipping split with only one training class:", seed, split_name)
            continue

        for model_name in MODELS:
            for text_col, version in TEXT_VERSIONS:

                # Body text may be empty for some rows
                train_data = train[
                    train[text_col].fillna("").str.strip() != ""
                ]
                test_data = test[
                    test[text_col].fillna("").str.strip() != ""
                ]

                if len(train_data) == 0 or len(test_data) == 0:
                    continue

                if train_data["y"].nunique() < 2:
                    continue

                scores = train_and_test(
                    model_name,
                    train_data,
                    test_data,
                    text_col
                )

                results.append({
                    "model": model_name,
                    "text": version,
                    "split": split_name,
                    "seed": seed,
                    **scores
                })


# --------------------------------------------------
# 9. Summarize results
# --------------------------------------------------

res = pd.DataFrame(results)

if res.empty:
    print("No results were produced. Check the dataset and splits.")
else:
    summary = (
        res.groupby(["model", "text", "split"])[
            ["acc", "prec", "rec", "f1"]
        ]
        .agg(["mean", "std"])
        .round(3)
    )

    print("\nAverage results and standard deviations:")
    print(summary)

    f1_table = res.groupby(
        ["model", "text", "split"]
    )["f1"].mean().unstack()

    if "random" in f1_table.columns and "held-out" in f1_table.columns:
        f1_table["f1_drop"] = (
            f1_table["random"] - f1_table["held-out"]
        )

    print("\nF1 scores and the drop from random to held-out:")
    print(f1_table.round(3))

    res.to_csv("results_all_runs.csv", index=False)
    summary.to_csv("results_summary.csv")

    print("\nSaved results_all_runs.csv and results_summary.csv")

    # Size and fake-news share of the test sets (held-out test sets vary more)
    print("\nTest set size and fake rate (mean and std over seeds):")
    print(res.groupby("split")[["n_test", "test_fake_rate"]].agg(["mean", "std"]).round(3))

    # Confusion matrix counts summed over all seeds (Logistic Regression, body_masked)
    cm = res[
        (res["model"] == "Logistic Regression") & (res["text"] == "body_masked")
    ].groupby("split")[["tn", "fp", "fn", "tp"]].sum()
    print("\nConfusion counts summed over seeds (Logistic Regression, body_masked):")
    print(cm)
